In [ ]:
'''
Both WL and NBI imaging technologies are included
Metadata provided by the gastroenterologist
1. Number of polyps of interest found during the procedure;
2. Current polyp ID;
3. Polyp size, in millimetres;
4. Paris classification [26,27];
5. NICE classification [29];
6. Preliminary diagnosis
The pathologist completed the metadata:
1. Final diagnosis;
2. Histological stratification.

Video clips were processed to extract the individual frames.
All frames were analysed and categorized into showing a polyp or not, as well as identifying the type of light source (WL or NBI)
76 lesions from 40 patients
62 out of these 76 lesions included white light (WL) and narrow band imaging (NBI) frames, while the remaining 14 lesions are recorded only using WL

'''

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
import cv2
import pandas as pd
from PIL import Image, ImageDraw
from scipy import ndimage as ndi
import tkinter as tk
from tkinter import filedialog
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from common import crop_image_padding, move_images
from sklearn.cluster import KMeans
import shutil

In [ ]:
# Open file dialog to select the spreadsheet file
csv_path = filedialog.askopenfilename(
    title="Select the csv file",
    filetypes=[("CSV files", "*.csv")]
)

In [ ]:
# Attempting to read the CSV file with a different encoding, in case of encoding issues
try:
    piccolo_df = pd.read_csv(csv_path, encoding='ISO-8859-1', delimiter=';')
except Exception as e:
    print("Error reading file:", e)

In [ ]:
# Replacing "-" placeholder values with NaN
piccolo_df = piccolo_df.replace('-', pd.NA)
# Dropping rows where all elements are NaN
piccolo_df = piccolo_df.dropna(how='all')
# Dropping columns where all elements are NaN
piccolo_df = piccolo_df.dropna(axis=1, how='all')

In [ ]:
columns_to_drop = ['HISTOLOGICAL STRATIFICATION', 'LITERAL DIAGNOSIS (Pathologist)', 
                             'PRELIMINAR DIAGNOSIS\xa0 (Gastroenterologist)', 'NICE ']

# Dropping the correctly named columns
piccolo_df = piccolo_df.drop(columns=columns_to_drop)

In [ ]:
# Code version formatting
# Handling NaN values before converting 'CODE - LESION' to a three-digit code format
piccolo_df['CODE - LESION'] = piccolo_df['CODE - LESION'].fillna(0).apply(lambda x: '{:03d}'.format(int(x)))

In [ ]:
# Converting 'NUMBER OF POLYPS OF INTEREST (N)' to an integer value
piccolo_df['NUMBER OF POLYPS OF INTEREST (N)'] = piccolo_df['NUMBER OF POLYPS OF INTEREST (N)'].fillna(0).astype(int)

In [ ]:
# Define the base path where the folders are located
base_path = os.path.dirname(csv_path)

# New DataFrame structure
new_frame_data = []

# Iterate through the 'test', 'train', and 'validation' folders
for set_name in ['test', 'train', 'validation']:
    polyp_folder_path = os.path.join(base_path, set_name, 'polyps')
    
    # Check if the mask folder exists
    if os.path.exists(polyp_folder_path) and os.path.isdir(polyp_folder_path):
        # List all frame files in the directory
        for frame_file in os.listdir(polyp_folder_path):
            if frame_file.endswith('.png'):
                code_lesion_str, code_video, _ = frame_file.split('_')
                # code_lesion = float(code_lesion_str.lstrip('0') or '0')
                matching_row = piccolo_df[(piccolo_df['CODE - LESION'] == code_lesion_str) & 
                                           (piccolo_df['CODE - VIDEO'] == code_video)]

                # If a matching row is found in the original dataframe
                if not matching_row.empty:
                    # Extract the row as a dictionary and add the frame file info
                    frame_data = {'FRAME': frame_file}
                    frame_data['SET'] = set_name
                    # Add other data from the matching row
                    for col in piccolo_df.columns[:-1]:
                        frame_data[col] = matching_row.iloc[0][col]
                    
                    new_frame_data.append(frame_data)
                else:
                    print('No matching row for ', frame_file, ' in ', set_name)

# Create the new DataFrame
extended_piccolo_df = pd.DataFrame(new_frame_data)

In [ ]:
extended_piccolo_df = extended_piccolo_df.dropna(subset=['PARIS CLASSIFICATION'])

In [ ]:
# Rename the column 'PARIS CLASSIFICATION' to 'SHAPE'
extended_piccolo_df = extended_piccolo_df.rename(columns={'PARIS CLASSIFICATION': 'SHAPE'})

In [ ]:
# First, replace 'l' with 'I' to correct the codes
extended_piccolo_df['SHAPE'] = extended_piccolo_df['SHAPE'].str.replace('l', 'I', regex=False).str.replace('Ips', 'Isp', regex=False)

# Define a regular expression pattern that matches the target codes
# The pattern looks for the codes, accounting for potential variations and corrections made
pattern = r'(Ip|Isp|Is|IIa|IIb|IIc|III)'

# Use str.extract to find matches of the pattern in the 'SHAPE' column and replace the column with the extracted codes
extended_piccolo_df['SHAPE'] = extended_piccolo_df['SHAPE'].str.extract(pattern)

polyp_pathology_distribution = extended_piccolo_df['SHAPE'].value_counts()
polyp_pathology_distribution

In [ ]:
# Cropping padding

In [ ]:
output_frame_folder_base = os.path.join(base_path, 'cropped_polyps')
os.makedirs(output_frame_folder_base, exist_ok=True)
output_mask_folder_base = os.path.join(base_path, 'cropped_masks')
os.makedirs(output_mask_folder_base, exist_ok=True)

In [ ]:
for index, row in extended_piccolo_df.iterrows():
    frame_path = os.path.join(base_path, row['SET'], 'polyps', row['FRAME'])
    mask_path = os.path.join(base_path, row['SET'], 'masks', row['FRAME'].replace('.png', '_Corrected.tif'))
    cropped_frame_path = os.path.join(output_frame_folder_base, row['FRAME'])
    cropped_mask_path = os.path.join(output_mask_folder_base, row['FRAME'].replace('.png', '.tif'))
    crop_image_padding(frame_path, cropped_frame_path, mask_path=mask_path, cropped_mask_path=cropped_mask_path, padding_color=(15,15,15), threshold=20)

In [ ]:
# Adding width and height columns to the DataFrame
extended_piccolo_df['WIDTH'] = None
extended_piccolo_df['HEIGHT'] = None

for index, row in extended_piccolo_df.iterrows():
    frame_path = os.path.join(output_frame_folder_base, row['FRAME'])
    try:
        with Image.open(frame_path) as img:
            width, height = img.size
            extended_piccolo_df.at[index, 'WIDTH'] = width
            extended_piccolo_df.at[index, 'HEIGHT'] = height
            
    except IOError:
        print(f"Unable to open file: {frame_path}")

extended_piccolo_df

In [ ]:
extended_piccolo_df.drop(columns=['CODE - LESION', 'CODE - VIDEO', 'NUMBER OF POLYPS OF INTEREST (N)', 'CURRENT POLYP ID (X/N)', 'SET'], inplace=True)

In [ ]:
# Figure out the image type: WL or NBI

extended_piccolo_df['IMAGE TYPE'] = None
extended_piccolo_df.reset_index(drop=True, inplace=True)  # Reset the index

features = []
indices_clean = []  # Store indices of successfully processed images

def extract_color_histogram(image):
    # Convert image to RGB (OpenCV loads images in BGR)
    image = cv2.cvtColor(np.array(image), cv2.COLOR_BGR2RGB)
    # Compute the color histogram
    hist = cv2.calcHist([image], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256])
    # Normalize the histogram
    hist = cv2.normalize(hist, hist).flatten()
    return hist

for index, row in extended_piccolo_df.iterrows():
    frame_path = os.path.join(output_frame_folder_base, row['FRAME'])
    try:
        with Image.open(frame_path) as img:
            # Extract color histogram features for each image
            hist_features = extract_color_histogram(img)
            features.append(hist_features)
            indices_clean.append(index)  # Append index only on success
    except IOError:
        print(f"Unable to open file: {frame_path}")

# Apply k-means clustering to the cleaned features
kmeans = KMeans(n_clusters=2, random_state=0, n_init=10).fit(features)
labels = kmeans.labels_

# Add the classification results to a new column in the DataFrame
for i, label in zip(indices_clean, labels):
    image_type = 'NBI' if label == 0 else 'WL'
    extended_piccolo_df.at[i, 'IMAGE TYPE'] = image_type

In [ ]:
# Function to calculate the bounding box of a mask
def calculate_bounding_box(mask):
    # Find all non-zero points (where the mask is)
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    ymin, ymax = np.where(rows)[0][[0, -1]]
    xmin, xmax = np.where(cols)[0][[0, -1]]

    # Check if the bounding box is at least 2x2 pixels
    if (ymax - ymin) > 1 and (xmax - xmin) > 1:
        return f"{xmin},{ymin},{xmax},{ymax}"
    else:
        return None


# Adding bounding box column to the DataFrame
extended_piccolo_df['BOUNDING BOX'] = None

for index, row in extended_piccolo_df.iterrows():
    mask_path = os.path.join(output_mask_folder_base, row['FRAME'].replace('.png', '.tif'))
    try:
        with Image.open(mask_path) as img:
            mask = np.array(img)
            
            # Assuming the mask is binary (0 or 255)
            mask = mask > 0
            
            # Calculate bounding box for each connected component (mask)
            bounding_boxes = []
            # Labeling connected components in the mask
            labeled, num_labels = ndi.label(mask)
            for label_num in range(1, num_labels + 1):
                label_mask = labeled == label_num
                bbox = calculate_bounding_box(label_mask)
                if bbox:
                    bounding_boxes.append(bbox)
            
            # Join all bounding box strings with a space
            extended_piccolo_df.at[index, 'BOUNDING BOX'] = ' '.join(bounding_boxes)
    except IOError:
        print(f"Unable to open file: {mask_path}")

extended_piccolo_df

In [ ]:
entries_with_multiple_bboxes = extended_piccolo_df[extended_piccolo_df['BOUNDING BOX'].str.count(',') > 3]
entries_with_multiple_bboxes

In [ ]:
# Filtering out entries with no bounding box
extended_piccolo_df = extended_piccolo_df[extended_piccolo_df['BOUNDING BOX'] != '']

In [ ]:
# Function to draw bounding boxes on an image
def draw_bounding_boxes(image_path, bounding_boxes, save_path):
    # Open the image
    with Image.open(image_path) as img:
        draw = ImageDraw.Draw(img)

        # Draw each bounding box
        for bbox_str in bounding_boxes.split(' '):
            coordinates = [int(coord.strip()) for coord in bbox_str.split(',') if coord.strip()]
            # print(coordinates)
            draw.rectangle(coordinates, outline="red", width=2)
        
        # Display the image
        # plt.imshow(img)
        # plt.show()
        
        # Save the image with bounding boxes
        img.save(save_path)

save_base_path = os.path.join(base_path, 'img_bbox')

# Create the save directory if it doesn't exist
os.makedirs(save_base_path, exist_ok=True)

# Loop through the DataFrame and draw bounding boxes on each image
for index, row in extended_piccolo_df.iterrows():
    frame_path = os.path.join(output_frame_folder_base, row['FRAME'])
    save_path = os.path.join(save_base_path, f"bbox_{row['FRAME']}") 
    draw_bounding_boxes(frame_path, row['BOUNDING BOX'], save_path)


In [ ]:
column_data_types = extended_piccolo_df.dtypes
column_data_types

In [ ]:
extended_piccolo_df = extended_piccolo_df.rename(columns={'POLYP SIZE (mm)': 'SIZE'})

In [ ]:
# Converting 'POLYP SIZE (mm)' to numeric (integer)
extended_piccolo_df.loc[:, 'SIZE'] = pd.to_numeric(extended_piccolo_df['SIZE'], errors='coerce')

# Assuming 'CODE - LESION' is a categorical identifier, we can leave it as an object (string). The same applies to 'CODE - VIDEO', 'PARIS CLASSIFICATION', and 'BOUNDING BOX'

# Converting 'WIDTH' to numeric (integer)
extended_piccolo_df.loc[:, 'WIDTH'] = pd.to_numeric(extended_piccolo_df['WIDTH'], errors='coerce')
# Converting 'HEIGHT' to numeric (integer)
extended_piccolo_df.loc[:, 'HEIGHT'] = pd.to_numeric(extended_piccolo_df['HEIGHT'], errors='coerce')

In [ ]:
# Distribution of polyp sizes, Histogram

In [ ]:
# Calculating the distribution of polyp sizes
polyp_size_distribution = extended_piccolo_df['SIZE'].value_counts()

# Plotting the histogram of polyp sizes
plt.figure(figsize=(10, 6))
extended_piccolo_df['SIZE'].hist(bins=range(0, int(extended_piccolo_df['SIZE'].max()) + 1, 1))
plt.title('Histogram of Polyp Sizes')
plt.xlabel('Polyp Size (mm)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

polyp_size_distribution

In [ ]:
# Descriptive statistics and Box plot

In [ ]:
# Calculating descriptive statistics for polyp sizes
polyp_size_descriptive_stats = extended_piccolo_df['SIZE'].describe()

# Plotting the box plot for polyp sizes
plt.figure(figsize=(8, 6))
extended_piccolo_df['SIZE'].plot(kind='box')
plt.title('Box Plot of Polyp Sizes')
plt.ylabel('Polyp Size (mm)')
plt.grid(True)
plt.show()

polyp_size_descriptive_stats

In [ ]:
# Distribution of polyp sizes based on the different ranges considered by POLAR's authors: Diminutive Polyps (1-5 mm), Small Polyps (6-9 mm), Larger Polyps (>=10 mm)

In [ ]:
def classify_polyp(size):
    if size <= 5:
        return 'Diminutive Polyps (1-5 mm)'
    elif size <= 9:
        return 'Small Polyps (6-9 mm)'
    else:
        return 'Larger Polyps (>=10 mm)'
    
polyp_classification = extended_piccolo_df['SIZE'].apply(classify_polyp)
polyp_size_distribution = polyp_classification.value_counts()


# Plotting the histogram
plt.figure(figsize=(10, 6))
polyp_size_distribution.plot(kind='bar')
plt.title('Histogram of Polyp Size Distribution')
plt.xlabel('Polyp Size Category')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.show()

polyp_size_distribution

In [ ]:
# Distribution by morphology (defined using the values considered by the Paris classification)

In [ ]:
# Calculating the distribution of polyps by Paris classification
paris_classification_distribution = extended_piccolo_df['SHAPE'].value_counts()

# Plotting the histogram
plt.figure(figsize=(10, 6))
paris_classification_distribution.plot(kind='bar')
plt.title('Distribution of Polyps by Paris Classification')
plt.xlabel('Paris Classification Category')
plt.ylabel('Frequency')
plt.xticks(rotation=45)
plt.show()

paris_classification_distribution

In [ ]:
# Polyp localization analysis based on bounding boxes. Descriptive statistics

In [ ]:
nan_rows = extended_piccolo_df[extended_piccolo_df['BOUNDING BOX'].isna()]
print(nan_rows)

In [ ]:
def extract_coordinates(bbox_str):
    print(bbox_str)
    coordinates = []
    if bbox_str:
        boxes = bbox_str.split(' ')
        for box in boxes:
            xmin, ymin, xmax, ymax = [int(coord) for coord in box.split(',')]
            coordinates.append((xmin, ymin, xmax, ymax))
    return coordinates

# Apply the function to the DataFrame
bbox_data = extended_piccolo_df['BOUNDING BOX'].apply(extract_coordinates)

# Initialize lists to store individual coordinates
min_x, min_y, max_x, max_y = [], [], [], []

# Extract individual coordinates
for row in bbox_data:
    for (xmin, ymin, xmax, ymax) in row:
        min_x.append(xmin)
        min_y.append(ymin)
        max_x.append(xmax)
        max_y.append(ymax)

# Calculate descriptive statistics for each set of coordinates
stats = {
    'min_x': pd.Series(min_x).describe(),
    'min_y': pd.Series(min_y).describe(),
    'max_x': pd.Series(max_x).describe(),
    'max_y': pd.Series(max_y).describe()
}

stats

In [ ]:
# Saving the filtered DataFrame to a spreadsheet
save_path = os.path.join(base_path, 'annotations.xlsx')
extended_piccolo_df.to_excel(save_path, index=False)

In [ ]:
# image base path
source_image_directory = output_frame_folder_base
# destination path
destination_image_directory = base_path + '/img/'

move_images(source_image_directory, destination_image_directory, extended_piccolo_df)